# 第 7 週 實作｜積分技巧總整理與數值積分

四招學完了,但有些積分被<strong>證明</strong>算不出公式。這週轉向:不求公式、只求數字——而且要能保證那個數字有多準。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜手刻三種積分器,量它們的收斂階

梯形 $O(h^2)$、Simpson $O(h^4)$——理論說的。這格把三種方法寫出來,用 log-log 圖<strong>量出斜率</strong>,看理論對不對。


In [ ]:
def trapezoid(f, a, b, n):
    h = (b - a) / n
    s = (f(a) + f(b)) / 2
    for i in range(1, n):
        s += f(a + i*h)
    return h * s

def simpson(f, a, b, n):
    assert n % 2 == 0, "Simpson 需要偶數段"
    h = (b - a) / n
    s = f(a) + f(b)
    for i in range(1, n):
        s += (4 if i % 2 else 2) * f(a + i*h)
    return h * s / 3

def gauss2(f, a, b):
    """兩點 Gauss-Legendre,先做區間變換"""
    c, r = (a + b)/2, (b - a)/2
    t = 1/math.sqrt(3)
    return r * (f(c - r*t) + f(c + r*t))

# 測試:∫_0^1 e^x dx = e - 1
f, a, b = math.exp, 0.0, 1.0
exact = math.e - 1

print(f"{'n':>4} {'梯形誤差':>12} {'比值':>7} {'Simpson誤差':>13} {'比值':>7}")
pt = ps = None
ns = [2, 4, 8, 16, 32, 64]
et, es = [], []
for n in ns:
    e1 = abs(trapezoid(f, a, b, n) - exact)
    e2 = abs(simpson(f, a, b, n) - exact)
    et.append(e1); es.append(e2)
    r1 = f"{pt/e1:7.2f}" if pt else "      -"
    r2 = f"{ps/e2:7.2f}" if ps else "      -"
    print(f"{n:4d} {e1:12.4e} {r1} {e2:13.4e} {r2}")
    pt, ps = e1, e2

print(f"\nGauss 兩點(只用 2 次求值)誤差 = {abs(gauss2(f, a, b) - exact):.4e}")
print(f"對照 梯形 n=2 (3 次求值)   誤差 = {et[0]:.4e}")

hs = np.array([(b - a)/n for n in ns])
plt.loglog(hs, et, 'o-', label='trapezoid')
plt.loglog(hs, es, 's-', label='Simpson')
plt.loglog(hs, hs**2 * et[0]/hs[0]**2, 'k:', label='slope 2 ref')
plt.loglog(hs, hs**4 * es[0]/hs[0]**4, 'k--', label='slope 4 ref')
plt.xlabel('h'); plt.ylabel('|error|'); plt.legend(fontsize=8)
plt.title('Measured convergence orders')
plt.show()

for name, e in [('trapezoid', et), ('Simpson', es)]:
    slope = np.polyfit(np.log10(hs[:4]), np.log10(e[:4]), 1)[0]
    print(f"{name:10s} log-log 斜率 = {slope:.3f}")

In [ ]:
# TODO 學生練習:把 f 換成 lambda t: 1/(1+t*t),a=0, b=1(精確值 pi/4)
# 這次 Simpson 的誤差比值還是 16 嗎?如果不是,先別看下一個 Lab,自己猜猜為什麼

## Lab 2｜超收斂:理論能不能事先預測

觀念 6 說:若 $f'''(b)=f'''(a)$,Simpson 的 $h^4$ 主項會消失、變成 $O(h^6)$。這格<strong>先預測、再實測</strong>,檢驗理論。


In [ ]:
x = sp.Symbol('x', real=True)

def simpson(f, a, b, n):
    h = (b - a) / n
    s = f(a) + f(b)
    for i in range(1, n):
        s += (4 if i % 2 else 2) * f(a + i*h)
    return h * s / 3

cases = [
    ("1/(1+x^2) on [0,1]",  1/(1+x**2),    0, 1),
    ("sin x on [0,pi/2]",   sp.sin(x),     0, sp.pi/2),
    ("exp x on [0,1]",      sp.exp(x),     0, 1),
    ("sin x on [0,2pi]",    sp.sin(x),     0, 2*sp.pi),
]

for name, expr, a, b in cases:
    # --- 先預測 ---
    f3 = sp.diff(expr, x, 3)
    jump = sp.simplify(f3.subs(x, b) - f3.subs(x, a))
    predicted = "O(h^6) 超收斂,比值 ~64" if jump == 0 else "O(h^4) 標準,比值 ~16"
    # --- 再實測 ---
    fn = sp.lambdify(x, expr, 'math')
    av, bv = float(a), float(b)
    exact = float(sp.integrate(expr, (x, a, b)))
    errs = [abs(simpson(fn, av, bv, n) - exact) for n in [8, 16, 32]]
    ratios = [errs[0]/errs[1], errs[1]/errs[2]]
    print(f"{name:22s}  f'''(b)-f'''(a) = {str(jump):>12s}")
    print(f"    預測: {predicted}")
    print(f"    實測比值: {ratios[0]:8.2f}, {ratios[1]:8.2f}\n")

In [ ]:
# TODO 學生練習:找一個你自己的 f 和區間,讓 f'''(b) = f'''(a)
# 提示:任何在 [0, 2*pi] 上的三角函數都符合。先預測,再實測驗證

## Lab 3｜手刻 vs scipy:什麼時候該用現成的

觀念 9 說 <code>quad</code> 是自適應的。這格用一個「尖峰」函數,看等距法和自適應法的差距有多大。


In [ ]:
from scipy.integrate import quad

def simpson(f, a, b, n):
    h = (b - a) / n
    s = f(a) + f(b)
    for i in range(1, n):
        s += (4 if i % 2 else 2) * f(a + i*h)
    return h * s / 3

# 一個又窄又高的尖峰:只有 x ~ 0 附近有值
spike = lambda t: math.exp(-1000 * t * t)
exact = math.sqrt(math.pi / 1000)          # ∫_-inf^inf,區間夠寬時近似成立

print("被積函數 exp(-1000 x^2) 在 [-1, 1] 上(尖峰只有 ~0.1 寬)")
print(f"{'方法':>22} {'結果':>16} {'誤差':>12} {'函數求值次數':>14}")
for n in [50, 200, 1000, 5000]:
    v = simpson(spike, -1, 1, n)
    print(f"{'等距 Simpson n=' + str(n):>22} {v:16.10f} {abs(v-exact):12.3e} {n+1:14d}")

v, err = quad(spike, -1, 1)
print(f"{'scipy quad(自適應)':>22} {v:16.10f} {abs(v-exact):12.3e} {'~200':>14}")
print(f"   quad 自己回報的誤差估計 = {err:.2e}")

xs = np.linspace(-1, 1, 1000)
plt.plot(xs, np.exp(-1000*xs**2))
plt.title('exp(-1000 x^2): almost all the area lives near 0')
plt.xlabel('x'); plt.show()

print("\n→ 等距法把大部分求值浪費在函數幾乎為零的地方;")
print("  自適應法自動把節點集中到尖峰,少算很多還更準。")

In [ ]:
# TODO 學生練習:把尖峰改成 exp(-100000 * t * t)(更窄)
# 等距 Simpson 需要多大的 n 才追得上 quad?quad 的誤差估計變了嗎?